# Phase 8: Explainable Recommendation Analysis Layer
## Factual, Evidence-Based Explanation Generation for Movie Recommendations

---

### 1. Title & Objective
This notebook presents the design and execution of the **Explainable Recommendation Analysis Layer** (`RecommendationExplainer`) developed in Phase 8.

**Objectives**:
1. Answer *"Why was this movie recommended?"* using factual model evidence rather than generic AI text.
2. Extract explicit metadata overlaps (shared genres, keywords, director, cast, top TF-IDF terms).
3. Decompose collaborative filtering contributions ($S(m, c) \times r(u, m)$) from user's positive rating history.
4. Decompose hybrid scores into weighted normalized content and collaborative contributions.

### 2. Why Explainable Recommendations?
Black-box recommendation systems output score lists without exposing *why* a particular item was ranked high. Explainability builds user trust, enables debugging, and exposes underlying feature relationships.

> [!IMPORTANT]
> **Fact-Based Explanation Policy**: The explainer system NEVER fabricates reasons such as *"People like you enjoyed this"* or *"This movie is trending"* unless directly backed by underlying model matrices and interaction logs.

### 3. Existing Recommendation Architecture
Phase 8 wraps around existing Phase 1–7 recommenders:
- **Content Model (Phase 4)**: TF-IDF feature space + weighted user profile vector $\mathbf{u} = \frac{\sum w_i \mathbf{v}_i}{\sum |w_i|}$.
- **Collaborative Model (Phase 6)**: Item-User sparse matrix + Item-Item Cosine Similarity matrix $S$.
- **Hybrid Model (Phase 7)**: Candidate pool union ($N_{\text{cand}}=100$), Min-Max score normalization, and weighted linear fusion $\text{score}_{\text{hybrid}} = \alpha \cdot \text{norm\_content} + (1 - \alpha) \cdot \text{norm\_cf}$.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Robust project root resolution for notebook environment or project root CWD
current_dir = Path(os.getcwd())
project_root = current_dir if (current_dir / "data").exists() else current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.collaborative_filter import ItemBasedCollaborativeRecommender
from src.personalizer import PersonalizedRecommender
from src.hybrid_recommender import HybridMovieRecommender
from src.explanation import RecommendationExplainer

print("Modules imported successfully.")

### 4. Load Existing Models/Data
We load TMDB 5000 clean movie tags and MovieLens latest-small interaction dataset.

In [ ]:
current_dir = Path(os.getcwd())
project_root = current_dir if (current_dir / "data").exists() else current_dir.parent
tmdb_clean_path = project_root / "data" / "processed" / "clean_movies.csv"
ml_dir = project_root / "data" / "raw" / "movielens" / "ml-latest-small"
ratings_path = ml_dir / "ratings.csv"
movies_path = ml_dir / "movies.csv"

clean_movies_df = pd.read_csv(tmdb_clean_path)
ratings_df = pd.read_csv(ratings_path)
movielens_movies_df = pd.read_csv(movies_path)

print(f"TMDB Clean Movies: {len(clean_movies_df):,} rows")
print(f"MovieLens Ratings: {len(ratings_df):,} rows")
print(f"MovieLens Movies:  {len(movielens_movies_df):,} rows")

### 5. Build a User Preference Example & Instantiate Explainer
We fit the Item-Based CF engine and compose the Hybrid Recommender.

In [ ]:
cf_rec = ItemBasedCollaborativeRecommender(
    min_ratings_per_movie=5,
    min_ratings_per_user=5,
    movies_df=movielens_movies_df
)
cf_rec.fit(ratings_df)

personalizer = PersonalizedRecommender(clean_movies_df)

hybrid_rec = HybridMovieRecommender(
    collaborative_recommender=cf_rec,
    personalizer=personalizer,
    movies_df=movielens_movies_df,
    tmdb_df=clean_movies_df
)
hybrid_rec.fit(ratings_df)

explainer = RecommendationExplainer(
    hybrid_recommender=hybrid_rec,
    movielens_movies_df=movielens_movies_df
)
print("RecommendationExplainer initialized successfully.")

### 6. Generate Hybrid Recommendations
We generate Top-5 recommendations for sample User 1 under $\alpha = 0.50$.

In [ ]:
sample_user_id = 1
top_recs = hybrid_rec.recommend(user_id=sample_user_id, alpha=0.5, top_n=5)
recs_df = pd.DataFrame(top_recs)
print(f"Top-5 Hybrid Recommendations for User {sample_user_id}:")
print(recs_df[['movieId', 'title', 'hybrid_score', 'norm_content_score', 'norm_cf_score']].to_string(index=False))

### 7. Explain Content Evidence
We inspect content metadata overlaps (genres, keywords, director, cast, top TF-IDF terms) for a recommendation.

In [ ]:
target_item = top_recs[0]
target_movie_id = target_item['movieId']
target_title = target_item['title']

print(f"--- Explaining Content Evidence for '{target_title}' ---")
full_exp = explainer.explain_hybrid_recommendation(user_id=sample_user_id, target_movie_id=target_movie_id, alpha=0.5)
content_ev = full_exp['content_evidence']
print(f"Available          : {content_ev['available']}")
print(f"Shared Genres      : {content_ev['shared_genres']}")
print(f"Shared Keywords    : {content_ev['shared_keywords'][:5]}")
print(f"Shared Directors   : {content_ev['shared_directors']}")
print(f"Shared Cast        : {content_ev['shared_cast']}")
print(f"Top TF-IDF Terms   : {content_ev['top_tfidf_terms']}")

### 8. Explain Collaborative Evidence
We extract contributing rated movies from user's history and their calculated similarity $\times$ rating contributions.

In [ ]:
print(f"--- Explaining Collaborative Evidence for '{target_title}' ---")
cf_ev = full_exp['cf_evidence']
print(f"Available                   : {cf_ev['available']}")
print(f"Raw CF Score                : {cf_ev['raw_cf_score']}")
print(f"Total Contributing Movies   : {cf_ev['total_contributing_movies']}")
print("Top Contributing Rated Movies:")
contrib_df = pd.DataFrame(cf_ev['similar_rated_movies'])
if not contrib_df.empty:
    print(contrib_df[['movieId', 'title', 'user_rating', 'similarity', 'contribution']].to_string(index=False))
else:
    print("No positive collaborative neighbors found.")

### 9. Explain Hybrid Score & Human-Readable Summary
Decomposing weighted hybrid score and displaying generated human-readable summary.

In [ ]:
hy_ev = full_exp['hybrid_evidence']
print(f"--- Score Decomposition for '{target_title}' (alpha={hy_ev['alpha']}) ---")
print(f"Normalized Content Score     : {hy_ev['normalized_content_score']:.6f}")
print(f"Normalized CF Score          : {hy_ev['normalized_cf_score']:.6f}")
print(f"Weighted Content Contrib (alpha) : {hy_ev['weighted_content_contribution']:.6f}")
print(f"Weighted CF Contrib (1 - alpha)  : {hy_ev['weighted_cf_contribution']:.6f}")
print(f"Final Hybrid Score           : {hy_ev['hybrid_score']:.6f}")
print(f"Dominant Branch              : {hy_ev['dominant_branch']}")
print(f"\nHuman-Readable Summary:\n{full_exp['summary']}")

### 10. Compare Strong vs Weak Recommendations
Comparing explanations across different candidate ranks.

In [ ]:
for rank, rec in enumerate(top_recs[:3], 1):
    exp = explainer.explain_hybrid_recommendation(user_id=sample_user_id, target_movie_id=rec['movieId'], alpha=0.5)
    print(f"\nRank {rank}: '{rec['title']}' (Hybrid Score: {rec['hybrid_score']:.4f})")
    print(f"Summary: {exp['summary']}")

### 11. Edge Cases & Limitations
1. **Unmapped Items**: Unmapped MovieLens items return `available=False` for content evidence without crashing.
2. **Cold-Start Disclaimer**: Explainability exposes available recommendation signals gracefully, but does **NOT** solve true new-user cold start because content preference profiles require user rating history.
3. **Data Leakage Safeguard**: Explanations strictly use historical training ratings. Future held-out evaluation ratings are NEVER included.

### 12. Conclusion
Phase 8 completes the Explainable Recommendation Analysis Layer, delivering transparent, evidence-based explanations derived directly from actual model outputs.